In [ ]:
import csv
import pandas as pd
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore') # Does not work for this kind of warning
pd.set_option('display.max_rows', None)


In [ ]:
#Perfect Shift Equity: Day shifts and Night calls are split evenly among all staff.
#Strict 40-Hour Workweeks: Tracks hourly totals week-by-week.
#24-Hour Post-Night Rest Rule: Workers are completely protected after a night call.
#Weekday-Only Clinic Blocks: 8-hour clinic days are strictly assigned Monday through Friday.

def generate_perfect_hospital_schedule(start_date_str, total_weeks=5):
    # Define names (Can easily replace with actual name strings)
    workers = [f"Worker {i+1}" for i in range(8)]
    
    # Comprehensive global tracking metrics over the 6-month period
    global_counts = {
        w: {'Day_Shifts': 0, 'Night_Shifts': 0, 'Clinic_Days': 0, 'Total_Hours': 0} 
        for w in workers
    }
    
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    schedule_data = []
    last_night_worker = None

    # Step-by-step weekly processing loops
    for week_idx in range(total_weeks):
        week_start_date = start_date + timedelta(weeks=week_idx)
        
        # Local trackers for the current 7-day operational block
        weekly_hospital_counts = {w: 0 for w in workers}
        week_days_manifest = {}
        
        # 1. Schedule 12-Hour Hospital Shifts first (Mon - Sun)
        for day_offset in range(7):
            current_date = week_start_date + timedelta(days=day_offset)
            date_str = current_date.strftime("%Y-%m-%d")
            day_name = current_date.strftime("%A")
            
            week_days_manifest[date_str] = {'day_name': day_name, 'day_hospital': None, 'night_hospital': None, 'clinic_staff': []}
            
            # --- Assign Day Shift ---
            # Filter: Check mandatory 24hr rest + cap at 2 hospital shifts max per week
            available_for_day = [
                w for w in workers 
                if w != last_night_worker and weekly_hospital_counts[w] < 2
            ]
            # Equity Sort: Prioritize by fewest local shifts, then fewest global Day shifts
            available_for_day.sort(key=lambda w: (weekly_hospital_counts[w], global_counts[w]['Day_Shifts'], global_counts[w]['Total_Hours']))
            assigned_day_worker = available_for_day[0]
            
            # Update Day Metrics
            week_days_manifest[date_str]['day_hospital'] = assigned_day_worker
            weekly_hospital_counts[assigned_day_worker] += 1
            global_counts[assigned_day_worker]['Day_Shifts'] += 1
            global_counts[assigned_day_worker]['Total_Hours'] += 12
            
            # --- Assign Night Shift (Night Call) ---
            # Filter: Cannot be today's day shift worker, cannot be yesterday's night worker, cap at 2 per week
            available_for_night = [
                w for w in workers 
                if w != assigned_day_worker and w != last_night_worker and weekly_hospital_counts[w] < 2
            ]
            # Equity Sort: Prioritize by fewest local shifts, then fewest global NIGHT calls explicitly
            available_for_night.sort(key=lambda w: (weekly_hospital_counts[w], global_counts[w]['Night_Shifts'], global_counts[w]['Total_Hours']))
            assigned_night_worker = available_for_night[0]
            
            # Update Night Metrics
            week_days_manifest[date_str]['night_hospital'] = assigned_night_worker
            weekly_hospital_counts[assigned_night_worker] += 1
            global_counts[assigned_night_worker]['Night_Shifts'] += 1
            global_counts[assigned_night_worker]['Total_Hours'] += 12
            
            # Save historical state for tomorrow's rest constraint rule
            last_night_worker = assigned_night_worker

        # 2. Backfill 8-Hour Clinic Days (Restricted to Monday - Friday only)
        for worker in workers:
            hospital_shifts_worked = weekly_hospital_counts[worker]
            required_clinic_days = 2 if hospital_shifts_worked == 2 else 5
            clinic_days_assigned = 0
            
            for date_str, day_data in week_days_manifest.items():
                if clinic_days_assigned == required_clinic_days:
                    break
                
                # Weekday Check: Clinics are strictly closed on weekends
                if day_data['day_name'] in ['Saturday', 'Sunday']:
                    continue
                
                # Structural Conflict & Rest Checks
                is_on_day_hospital = day_data['day_hospital'] == worker
                is_on_night_hospital = day_data['night_hospital'] == worker
                
                prev_date = datetime.strptime(date_str, "%Y-%m-%d") - timedelta(days=1)
                prev_date_str = prev_date.strftime("%Y-%m-%d")
                was_resting = False
                if prev_date_str in week_days_manifest:
                    was_resting = week_days_manifest[prev_date_str]['night_hospital'] == worker
                
                # If they are completely free on this weekday, log clinic duties
                if not is_on_day_hospital and not is_on_night_hospital and not was_resting:
                    day_data['clinic_staff'].append(worker)
                    clinic_days_assigned += 1
                    global_counts[worker]['Clinic_Days'] += 1
                    global_counts[worker]['Total_Hours'] += 8
                    
        # 3. Format and save rows chronologically
        for date_str, day_data in sorted(week_days_manifest.items()):
            schedule_data.append({
                'Date': date_str,
                'Day of Week': day_data['day_name'],
                'Day Shift (12hr)': day_data['day_hospital'],
                'Night Shift (12hr)': day_data['night_hospital'],
                '8hr Clinic Staff': ", ".join(day_data['clinic_staff'])
            })

    tab = pd.DataFrame(schedule_data)
    # 4. Export to clean structural spreadsheet
    csv_filename = "hospital_equitable_40hr_schedule.csv"
    fields = ['Date', 'Day of Week', 'Day Shift (12hr)', 'Night Shift (12hr)', '8hr Clinic Staff']
    with open(csv_filename, mode='w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(schedule_data)
        
    # Print the audited final metrics matrix
    print(f"Schedule successfully compiled and exported to {csv_filename}!\n")
    print("6-Month Cumulative Totals (Audited Multi-Tier Shift Equity Matrix):")
    print(f"{'Worker':<12} | {'Day Shifts':<12} | {'Night Calls':<12} | {'Clinic Days':<12} | {'Total Hours Logged':<18}")
    print("-" * 75)
    for worker, counts in sorted(global_counts.items()):
        print(f"{worker:<12} | {counts['Day_Shifts']:<12} | {counts['Night_Shifts']:<12} | {counts['Clinic_Days']:<12} | {counts['Total_Hours']:<18} hrs")
    return tab
    
# Run schedule engine starting tomorrow
dat = generate_perfect_hospital_schedule("2026-09-20")


In [ ]:
dat.columns = ['date','day_of_week','day_shift','night_shift','clinic']
dat['date'] = pd.to_datetime(dat['date'])
dat['week_no'] = (dat['date'] + pd.Timedelta(days=1)).dt.isocalendar().week
#dat['week_no'] = dat['date'].dt.isocalendar().week


In [ ]:
who = 'Worker 7'
worker = dat[dat[['day_shift','night_shift','clinic']].apply(lambda x: x.str.contains(who, case=False, na=False)).any(axis=1)]

worker['day_shift_hrs'] = 12
worker['night_shift_hrs'] = 12
worker['clinic_hrs'] = 0
for i in range(len(worker)):
    if worker['day_shift'].iloc[i] == who: 
        worker['day_shift_hrs'].iloc[i] = 12
    else:
        worker['day_shift_hrs'].iloc[i] = 0
        
    if worker['night_shift'].iloc[i] == who: 
        worker['night_shift_hrs'].iloc[i] = 12
    else:
        worker['night_shift_hrs'].iloc[i] = 0
    if who in worker.clinic.iloc[i]:
        worker['clinic_hrs'].iloc[i] = 8 
    else:
        worker['clinic_hrs'].iloc[i] = 0
worker['tot_hrs'] = worker['day_shift_hrs'] + worker['night_shift_hrs'] + worker['clinic_hrs']
worker.iloc[:20]
#dat.iloc[:20]
#worker.groupby('week_no')['tot_hrs'].sum()
dat

In [ ]:
dat.day_shift.value_counts()